# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
GPT_MODEL = "gpt-4.1-mini"
openai = OpenAI()

anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

ANTHROPIC_MODEL = "claude-sonnet-4-5-20250929"

anthropic_url = "https://api.anthropic.com/v1/"
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)


In [ ]:
system_prompt = """
You are a helpful assistant that analyzes a technical question,
and provides a short answer, with proper reason and method
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [52]:
def talker(reply):
    response = openai.audio.speech.create(model="gpt-4o-mini-tts",
      voice="alloy",
      input=reply)

    return response.content

In [80]:



def chat(history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role":"system","content":system_prompt}] + history

    stream = openai.chat.completions.create(model=GPT_MODEL,messages=messages,stream=True)
    for chunk in stream:
        result = chunk.choices[0].delta.content or 'hi'
        history += [{"role":"assistant","content":result}]
        voice = talker(result)
        yield history, voice



In [81]:
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=200, type="messages")
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=[chatbot], outputs=[chatbot, audio_output]
    )

In [82]:

ui.launch()

* Running on local URL:  http://127.0.0.1:7891
* To create a public link, set `share=True` in `launch()`.
